In [ ]:
import customtkinter as ctk
from tkinter import messagebox
from collections import deque
import random


ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

GOAL = (
    (1, 2, 3),
    (8, 0, 4),
    (7, 6, 5)
)

MOVES = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1)
}


def find_blank(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return None

def get_blankcanmoves(state):
    state_move = []
    x, y = find_blank(state)

    for move_name, (dx, dy) in MOVES.items():
        new_x, new_y = x + dx, y + dy
        if 0 <= new_x < 3 and 0 <= new_y < 3:
            matrix = [list(row) for row in state]
            matrix[x][y], matrix[new_x][new_y] = matrix[new_x][new_y], matrix[x][y]
            state_move.append(tuple(tuple(row) for row in matrix))

    return state_move

def get_move_direction(old_state, new_state):
    r1, c1 = find_blank(old_state)
    r2, c2 = find_blank(new_state)
    if r2 - r1 == 1: return "XUỐNG"
    if r1 - r2 == 1: return "LÊN"
    if c2 - c1 == 1: return "PHẢI"
    if c1 - c2 == 1: return "TRÁI"
    return "CHẠM"

def DFS(state):
    if state == GOAL:
        return [state], 1

    frontier = deque()
    frontier.append((state, [state]))

    reached = set()
    reached.add(state)

    while len(frontier) > 0:
        current_state, path = frontier.pop()
        state_move = get_blankcanmoves(current_state)

        for next_step in state_move:
            if next_step not in reached:
                if next_step == GOAL:
                    return path + [next_step], len(reached)

                reached.add(next_step)
                frontier.append((next_step, path + [next_step]))

    return None, len(reached)

def format_board(matrix):
    res = ""
    for row in matrix:
        res += "   " + "  ".join(str(val) if val != 0 else "_" for val in row) + "\n"
    return res


class PuzzleApp(ctk.CTk):
    def __init__(self):
        super().__init__()

        self.title("8-Puzzle DFS Dashboard")
        self.geometry("780x580")
        self.resizable(False, False)

        self.current_state = [list(row) for row in GOAL]
        self.grid_buttons = []
        self.player_steps = 0

        self.grid_columnconfigure(0, weight=2)
        self.grid_columnconfigure(1, weight=1)

        self.create_widgets()
        self.shuffle_board()

    def create_widgets(self):

        self.left_frame = ctk.CTkFrame(self, fg_color="transparent")
        self.left_frame.grid(row=0, column=0, sticky="nsew", padx=20, pady=20)

        ctk.CTkLabel(self.left_frame, text="8-PUZZLE DFS SOLVER", font=("Helvetica", 26, "bold")).pack(pady=10)

        self.board_frame = ctk.CTkFrame(self.left_frame, width=300, height=300, corner_radius=15, fg_color="#2b2b2b")
        self.board_frame.pack(pady=15)

        for i in range(3):
            button_row = []
            for j in range(3):
                btn = ctk.CTkButton(
                    self.board_frame,
                    text="",
                    font=("Helvetica", 28, "bold"),
                    width=85, height=85, corner_radius=10,
                    command=lambda r=i, c=j: self.player_click(r, c)
                )
                btn.grid(row=i, column=j, padx=6, pady=6)
                button_row.append(btn)
            self.grid_buttons.append(button_row)

        self.player_step_label = ctk.CTkLabel(self.left_frame, text="Số bước bạn đi: 0", font=("Helvetica", 15, "bold"), text_color="#f39c12")
        self.player_step_label.pack(pady=(0, 5))

        self.status_label = ctk.CTkLabel(self.left_frame, text="Trạng thái: Sẵn sàng!", font=("Helvetica", 14), text_color="#bdc3c7")
        self.status_label.pack(pady=(0, 10))

        self.control_frame = ctk.CTkFrame(self.left_frame, fg_color="transparent")
        self.control_frame.pack(pady=5)

        self.shuffle_btn = ctk.CTkButton(self.control_frame, text="Xáo Trộn", width=120, height=38, corner_radius=8, command=self.shuffle_board)
        self.shuffle_btn.grid(row=0, column=0, padx=10)

        self.solve_btn = ctk.CTkButton(self.control_frame, text="AI Giải (DFS)", width=120, height=38, corner_radius=8, fg_color="#27ae60", hover_color="#2ecc71", command=self.click_solve_dfs)
        self.solve_btn.grid(row=0, column=1, padx=10)


        self.right_frame = ctk.CTkFrame(self, width=280, corner_radius=15, fg_color="#1e1e1e", border_width=1, border_color="#333333")
        self.right_frame.grid(row=0, column=1, sticky="nsew", padx=(0, 20), pady=20)

        ctk.CTkLabel(self.right_frame, text="📊 DASHBOARD", font=("Helvetica", 18, "bold"), text_color="#3498db").pack(pady=(20, 15))


        card1 = ctk.CTkFrame(self.right_frame, corner_radius=10, fg_color="#2b2b2b")
        card1.pack(fill="x", padx=15, pady=(0, 15))

        ctk.CTkLabel(card1, text="Trạng thái đã duyệt (Reached)", font=("Helvetica", 12), text_color="#aaaaaa").pack(pady=(10,0))
        self.reached_val = ctk.StringVar(value="0")
        ctk.CTkLabel(card1, textvariable=self.reached_val, font=("Helvetica", 28, "bold"), text_color="#f39c12").pack(pady=(0, 10))


        card2 = ctk.CTkFrame(self.right_frame, corner_radius=10, fg_color="#2b2b2b")
        card2.pack(fill="x", padx=15, pady=(0, 15))

        ctk.CTkLabel(card2, text="Tiến trình AI giải", font=("Helvetica", 12), text_color="#aaaaaa").pack(pady=(10,0))
        self.step_val = ctk.StringVar(value="0 / 0")
        ctk.CTkLabel(card2, textvariable=self.step_val, font=("Helvetica", 24, "bold"), text_color="#2ecc71").pack(pady=(0, 5))

        self.progress_bar = ctk.CTkProgressBar(card2, height=8, progress_color="#2ecc71", fg_color="#404040")
        self.progress_bar.set(0)
        self.progress_bar.pack(fill="x", padx=20, pady=(0, 15))

        # --- Khung Log ---
        ctk.CTkLabel(self.right_frame, text="📝 Trạng thái bàn cờ", font=("Helvetica", 13, "bold")).pack(pady=(5,5))
        self.log_textbox = ctk.CTkTextbox(self.right_frame, font=("Consolas", 13), fg_color="#141414", corner_radius=10)
        self.log_textbox.pack(fill="both", expand=True, padx=15, pady=(0, 15))
        self.log_textbox.configure(state="disabled")

        self.update_buttons_display()


    def append_log(self, text):
        self.log_textbox.configure(state="normal")
        self.log_textbox.insert("end", text + "\n")
        self.log_textbox.see("end")
        self.log_textbox.configure(state="disabled")

    def clear_log(self):
        self.log_textbox.configure(state="normal")
        self.log_textbox.delete("1.0", "end")
        self.log_textbox.configure(state="disabled")

    def update_buttons_display(self):
        for i in range(3):
            for j in range(3):
                value = self.current_state[i][j]
                self.grid_buttons[i][j].configure(
                    text=str(value) if value != 0 else "",
                    fg_color="#1f538d" if value != 0 else "#1a252f",
                    hover_color="#2980b9" if value != 0 else "#1a252f"
                )

    def player_click(self, r, c):
        x_blank, y_blank = find_blank(tuple(tuple(row) for row in self.current_state))

        if abs(r - x_blank) + abs(c - y_blank) == 1:
            self.player_steps += 1
            self.player_step_label.configure(text=f"Số bước bạn đi: {self.player_steps}")

            old_state = tuple(tuple(row) for row in self.current_state)
            self.current_state[x_blank][y_blank], self.current_state[r][c] = self.current_state[r][c], self.current_state[x_blank][y_blank]

            move_dir = get_move_direction(old_state, tuple(tuple(row) for row in self.current_state))
            board_str = format_board(self.current_state)
            self.append_log(f"Bạn kéo: {move_dir}\n{board_str}")

            self.update_buttons_display()

            if tuple(tuple(row) for row in self.current_state) == GOAL:
                messagebox.showinfo("Chúc mừng", f"Bạn đã tự giải thành công trong {self.player_steps} bước!")

    def shuffle_board(self):
        self.player_steps = 0
        self.player_step_label.configure(text="Số bước bạn đi: 0")
        self.reached_val.set("0")
        self.step_val.set("0 / 0")
        self.progress_bar.set(0)

        self.clear_log()
        self.append_log("Đang xáo trộn...\n" + "-"*20)

        current = GOAL
        previous = None

        for _ in range(50):
            neighbors = get_blankcanmoves(current)
            valid_moves = [n for n in neighbors if n != previous]
            if not valid_moves: valid_moves = neighbors
            previous = current
            current = random.choice(valid_moves)

        self.current_state = [list(row) for row in current]
        self.update_buttons_display()
        self.status_label.configure(text="Trạng thái: Đã xáo trộn đề bài!")

        board_str = format_board(self.current_state)
        self.append_log(f"BẮT ĐẦU:\n{board_str}")

    def click_solve_dfs(self):
        self.status_label.configure(text="AI đang phân tích đường đi (DFS)...")
        self.shuffle_btn.configure(state="disabled")
        self.solve_btn.configure(state="disabled")
        self.update()

        start_tuple = tuple(tuple(row) for row in self.current_state)
        path_solution, reached_count = DFS(start_tuple)

        if path_solution is not None:
            steps = len(path_solution) - 1
            self.reached_val.set(f"{reached_count:,}")
            self.status_label.configure(text=f"Tìm thấy giải pháp: {steps} bước.")
            self.append_log(f"-> Giải pháp: {steps} bước\n" + "="*20)

            self.animate_solution(path_solution, 0, steps)
        else:
            self.status_label.configure(text="Không tìm thấy lời giải hợp lệ!")
            self.append_log("THẤT BẠI: Không có lời giải.")
            self.shuffle_btn.configure(state="normal")
            self.solve_btn.configure(state="normal")

    def animate_solution(self, path, cur_step, total_steps):
        if len(path) == 0:
            self.status_label.configure(text="Hoàn thành xuất sắc!", text_color="#2ecc71")
            self.shuffle_btn.configure(state="normal")
            self.solve_btn.configure(state="normal")
            return

        next_step = path.pop(0)
        old_state = tuple(tuple(row) for row in self.current_state)
        move_dir = get_move_direction(old_state, next_step)

        self.current_state = [list(row) for row in next_step]
        self.step_val.set(f"{cur_step} / {total_steps}")

        if total_steps > 0:
            self.progress_bar.set(cur_step / total_steps)

        if move_dir != "CHẠM":
            board_str = format_board(self.current_state)
            self.append_log(f"[Bước {cur_step:02d}]: {move_dir}\n{board_str}")

        self.update_buttons_display()


        self.after(150, lambda: self.animate_solution(path, cur_step + 1, total_steps))

if __name__ == "__main__":
    app = PuzzleApp()
    app.mainloop()